In [ ]:
import os
import re
import json
from dotenv import load_dotenv 
from openai import OpenAI
from pathlib import Path
import pymupdf
from tqdm import tqdm
import re
from tenacity import retry, wait_random_exponential, stop_after_attempt

In [ ]:
# Read API key from environment
load_dotenv()
client = OpenAI()
# Choose your model
model_ID = "gpt-5-mini"  

In [14]:

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

resp = client.responses.create(
    model="gpt-5",
    input="ping"
)
print(resp.output_text)

pong


In [ ]:
def read_prompt(prompt_path: str):
  with open(prompt_path, "r") as f:
    return f.read()

In [16]:
SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "PaperTitle",
        "PublicationYear",
        "DOI",
        "Authors",
        "Abstract",
        "SummaryAbstract",
        "References"
    ],
    "properties": {
        "PaperTitle": {
            "type": "string",
            "minLength": 0
        },
        "PublicationYear": {
            "type": "string",
            "pattern": r"^[0-9]{4}$|^$"
        },
        "DOI": {
            "type": "string",
            "minLength": 0
        },
        "Authors": {
            "type": "array",
            "items": {
                "type": "string",
                "minLength": 1
            }
        },
        "Abstract": {
            "type": "string",
            "minLength": 0
        },
        "SummaryAbstract": {
            "type": "string",
            "minLength": 0
        },
        "References": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "PaperTitle",
                    "Authors",
                    "PublicationYear",
                    "DOI",
                    "Publication"
                ],
                "properties": {
                    "PaperTitle": {
                        "type": "string",
                        "minLength": 0
                    },
                    "Authors": {
                        "type": "string",
                        "minLength": 0
                    },
                    "PublicationYear": {
                        "type": "string",
                        "pattern": r"^[0-9]{4}$|^$"
                    },
                    "DOI": {
                        "type": "string",
                        "minLength": 0
                    },
                    "Publication": {
                        "type": "string",
                        "minLength": 0
                    }
                }
            }
        }
    }
}


In [ ]:
def extract_text_from_pdf(pdf_path: str):
    doc = pymupdf.open(pdf_path)
    num_pages = len(doc)
    pages = []
      # Process each page of the PDF
    for page_num in tqdm(range(num_pages), desc="Processing PDF pages"):
        page = doc[page_num]
        pages.append(page.get_text("text"))
    doc.close()
    return "\n".join(pages)

In [ ]:
@retry(wait=wait_random_exponential(min=1, max=2400), stop=stop_after_attempt(10))
def completion_with_backoff(**kwargs):
  return client.chat.completions.create(**kwargs)


def extract_metadata(content: str, prompt_path: str, model_id: str):
  # Use GPT model to extract metadata from the research paper content based on the given prompt.
  #Read the prompt
  prompt_data = read_prompt(prompt_path)

  try:
    response = completion_with_backoff(
        model = model_id,
        messages = [
            {"role": "system", "content": prompt_data},
            {"role": "user", "content": content}
        ],
    )

    response_content = response.choices[0].message.content
    if not response_content:
      print("Empty response from the model")
      return {}

    #Remove any markdown code block indecators
    response_content = re.sub(r'```json\s*', '', response_content)
    response_content = re.sub(r'\s*```', '', response_content)

    #Attempt to pharse JSON
    try:
      return json.loads(response_content)
    except json.decoder.JSONDecodeError as e:
      print(f"Falied to parse JSON: {e}")
      print(f"Raw Response: {response_content}")

      #Attempt to extract JSON from the response
      match = re.search(r'\{.*\}', response_content, re.DOTALL)
      if match:
        try:
          return json.loads(match.group(0))
        except json.decoder.JSONDecodeError as jde:
          print(f"Failed to extrat valid JSON from the response: {jde}")

      return {}

  except Exception as e:
    print(f"Error calling OpenAI API: {e}")
    return {}



#process a single research paper
def process_research_paper(pdf_path: str, prompt: str, output_folder: str, model_id: str):
  # Process a single research paper through the entire pipeline.
  print(f"Processing research paper: {pdf_path}")

  try:
    #Step 1: Extract text content from the PDF
    content = extract_text_from_pdf(pdf_path)
    print(f"Extracted text content from the PDF: {pdf_path}")

    #Step 2: Use GPT
    metadata = extract_metadata(content, prompt, model_id)
    if not metadata:
      print(f"Failed to extract metadata from the PDF: {pdf_path}")
      return
    print(f"Extracted metadata using {model_id} for {pdf_path}")

    # Step 3: Save the result as a JSON file
    output_filename = Path(pdf_path).stem + ".json"
    output_path = os.path.join(output_folder, output_filename)

    with open(output_path, "w") as f:
      json.dump(metadata, f, indent=2)
    print(f"Saved metadata to {output_path}")

  except Exception as e:
    print(f"Error processing {pdf_path} research paper: {e}")

In [19]:
pdf_path = Path("./papers/Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.pdf").expanduser().resolve()
prompt_path = Path("./prompts/information_extraction_prompt.txt").expanduser().resolve()
output_folder = Path("./IE-output").expanduser().resolve()


process_research_paper(pdf_path, prompt_path, output_folder, model_ID)

Processing research paper: D:\Masters\Research Project\Graph-Theoretic-Research-Project\papers\Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.pdf


Processing PDF pages: 100%|██████████| 6/6 [00:00<00:00, 168.23it/s]

Extracted text content from the PDF: D:\Masters\Research Project\Graph-Theoretic-Research-Project\papers\Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.pdf


Extracted metadata using gpt-5-mini for D:\Masters\Research Project\Graph-Theoretic-Research-Project\papers\Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.pdf
Saved metadata to D:\Masters\Research Project\Graph-Theoretic-Research-Project\IE-output\Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.json


In [20]:
output_file = Path("./IE-output/Graph_Embedding_for_Mapping_Interdisciplinary_Research_Network.json").expanduser().resolve()
with open(output_file, "r", encoding="utf-8") as f:
    data = json.load(f)
    count = len(data["References"])
    print("Number of references objects:", count)

Number of references objects: 19
